# Stage 2 — screen

Re-create this stage's script with Gemini's help. The cells below give you the spec, the seed, the gotchas, and a verification step. The implementation itself is yours to write.


## 1. Setup

Every cell in this section is idempotent and safe to re-run. If you opened this notebook fresh (without running Stage 0 first in the same runtime), run all of them now.


### 1a. Clone the repo and `cd` into it


In [ ]:
# Bootstrap: clone the workshop repo into /content and cd into it.
# Idempotent — safe to re-run.
import os, subprocess, sys
REPO_DIR = "/content/ar-bic-2026-workshop"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/jayprimer/ar-bic-2026-workshop.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


### 1b. Install dependencies

Python (`openai`) and the Node CLI `@llamaindex/liteparse`. First run takes ~30s; re-runs are near-instant.


In [ ]:
# Install dependencies. Idempotent (pip skips already-installed; npm re-link is cheap).
# liteparse only matters for Stage 4 but installing it everywhere keeps each
# notebook self-contained, which is the whole point of re-running this cell.
!pip install -q -r requirements.txt
!npm install -g @llamaindex/liteparse 2>&1 | tail -3


### 1c. Bridge your OpenAI key

Add `OPENAI_API_KEY` in Colab's Secrets panel (key icon, left sidebar) and toggle notebook access first.


In [ ]:
# OpenAI key bridge: Colab's userdata.get() does NOT populate os.environ,
# but our scripts read os.environ["OPENAI_API_KEY"]. Bridge it once.
# Add the key in Colab via the left sidebar → "Secrets" (key icon) → name it OPENAI_API_KEY.
import os
try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
        print("OPENAI_API_KEY set in os.environ")
    else:
        print("WARNING: OPENAI_API_KEY secret is empty — Stage 2/5 and *_llm.py evals will fail")
except Exception as e:
    print("Not running in Colab or userdata unavailable; set OPENAI_API_KEY yourself.")
    print("Detail:", e)


### 1d. Stage the bundle-shipped configs


In [ ]:
# Copy bundle-shipped configs into the directories each stage script expects.
# Each stage's input.txt / criteria.txt / schema.json lives under configs/
# in the repo; the actual scripts read them relative to cwd.
import os, shutil
os.makedirs("stage_01", exist_ok=True)
os.makedirs("stage_02", exist_ok=True)
shutil.copy("configs/stage_01_input.txt",    "stage_01/input.txt")
shutil.copy("configs/stage_02_input.txt",    "stage_02/input.txt")
shutil.copy("configs/stage_02_criteria.txt", "stage_02/criteria.txt")
shutil.copy("configs/schema.json",           "schema.json")
print("configs staged")


### 1e. Load prior stages' reference outputs

Stage 2 reads outputs from earlier stages. Each Colab notebook gets its own runtime, so work done in another notebook is not visible here. This cell seeds `stage_01..stage_01/data/` from the canonical reference run so Stage 2 has inputs to work with.


In [ ]:
# Load prior stages' reference outputs as inputs for Stage 2.
# Each Colab notebook opens with a fresh runtime, so any work done in a
# Stage <2 notebook in a DIFFERENT runtime is not visible here.
# This cell makes the stage runnable in isolation against the canonical
# reference run. If you re-run an earlier stage IN THIS runtime, your
# output replaces these reference files (cwd is /content/...).
import os, shutil, glob
for n in range(1, 2):
    dst = f"stage_0{n}/data"
    src = f"reference_outputs/stage_0{n}/data"
    if not os.path.isdir(src):
        continue
    os.makedirs(dst, exist_ok=True)
    # Only seed if the participant hasn't produced anything for this stage
    # in the current runtime — otherwise we'd clobber their work.
    if any(os.scandir(dst)):
        print(f"skip stage_0{n} — already has files (keeping your work)")
        continue
    for src_file in glob.glob(f"{src}/*"):
        shutil.copy(src_file, dst)
    print(f"seeded stage_0{n}/data from reference_outputs")


## 2. Spec — paste this into Gemini

Open the Gemini side panel in Colab (sparkles icon, top right) and paste the block below as your prompt. Then iterate.

```
Write a Python script that:

1. Reads `stage_02/input.txt` (same KEY=value parser as Stage 1).
   Keys: MODEL (default gpt-5.4-nano), CRITERIA_FILE
   (default stage_02/criteria.txt), SLEEP_SECONDS (default 1.0).
2. Reads `stage_02/criteria.txt` as a plain-text inclusion/exclusion
   prompt.
3. Reads `stage_01/data/pmids.json` → `records` list.
4. For each record, calls OpenAI chat.completions with the criteria
   inlined into the prompt; asks for STRICT JSON
   `{"verdict": "include"|"exclude", "rationale": "<one sentence>"}`.
5. Writes `stage_02/data/screened.json` — a list, one object per
   record, carrying `pmid`, `title`, `abstract`, `pub_types`,
   `verdict`, `rationale`.

Use `temperature=0` and `response_format={"type": "json_object"}`.
```


## 3. Gotchas Gemini probably won't know

Copy any that apply into Gemini if it goes off-track:

- **Use the OpenAI JSON-mode response format.** Pass
  `response_format={"type": "json_object"}` AND `temperature=0` so the
  output is parseable without regex.
- **`OpenAI()` reads the env var.** As long as you've bridged the
  Colab secret in the setup cell above, no explicit api_key arg.
- **Rate-limit yourself.** Sleep `SLEEP_SECONDS` between requests
  (the free tier rate-limits aggressively).
- **Strip everything you don't need from the record before writing.**
  Carrying every metadata field forward bloats `screened.json`.


## 4. Seed — a few lines to anchor Gemini in the right direction


In [ ]:
import json, os, time
STAGE = "stage_02"
DATA = f"{STAGE}/data"
os.makedirs(DATA, exist_ok=True)
INPUT_PATH = f"{STAGE}/input.txt"

# config-file parser from Stage 1 (reuse — or ask Gemini to re-emit it)


## 5. Your implementation

Drive Gemini to fill this in. Iterate until the verification cell below passes.


In [ ]:
# TODO: implement Stage 2 here.
# Read the spec above. Use the seed cell's imports.
# When done, run the verification cell next.


## 6. Verify


In [ ]:
import json, os
assert os.path.exists("stage_02/data/screened.json"), "no output file"
rows = json.load(open("stage_02/data/screened.json"))
for r in rows:
    assert r["pmid"], "missing pmid"
    assert r["verdict"] in {"include", "exclude"}, r
    assert r["rationale"].strip(), "empty rationale"
print(f"OK — {sum(1 for r in rows if r['verdict']=='include')}/{len(rows)} included")


## 7. Run the eval grader

The eval reads only your stage's output and writes `stage_02/eval/eval_*.json` + `score.json`.


In [ ]:
!python eval/eval_02_script.py
# Optional (needs API key):
# !python eval/eval_02_llm.py


## 8. Stuck? Skip this stage

Copy the reference run's Stage 2 output into place so the next stage's notebook can still run. Use this sparingly — the point of the workshop is to *re-create* each stage.


In [ ]:
import os, shutil
os.makedirs("stage_02/data", exist_ok=True)
shutil.copy("reference_outputs/stage_02/data/screened.json",
            "stage_02/data/screened.json")
print("copied reference Stage 2 output")
